In [1]:
import torch

torch.manual_seed(123)
batch_example = torch.randn(2,5)
batch_example

tensor([[-0.1115,  0.1204, -0.3696, -0.2404, -1.1969],
        [ 0.2093, -0.9724, -0.7550,  0.3239, -0.1085]])

In [2]:
import torch.nn as nn

layer = nn.Sequential(
    nn.Linear(5,6),
    nn.ReLU()
)
out = layer(batch_example)
out

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)

#### Without layer Normalization

In [3]:
mean = out.mean(dim=-1, keepdim=True)   # (2,6) -> 6
var = out.var(dim=-1, keepdim=True)   
print("Mean: \n",mean)
print("Variance: \n",var)

Mean: 
 tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Variance: 
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


#### Using layer Normalization

In [4]:
out_normalized = (out - mean) / torch.sqrt_(var)

mean_norm = out_normalized.mean(dim=-1, keepdim=True)   # (2,6) -> 6
var_norm = out_normalized.var(dim=-1, keepdim=True)   

print("Normalized Mean: \n",mean_norm)
print("Normalized Variance: \n",var_norm)

Normalized Mean: 
 tensor([[9.9341e-09],
        [5.9605e-08]], grad_fn=<MeanBackward1>)
Normalized Variance: 
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


# Implemented Class

In [5]:
class LayerNormalize(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(embedding_dim))
        self.shift = nn.Parameter(torch.zeros(embedding_dim))
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)
        x_normalized = (x - mean) / torch.sqrt_(var + self.eps)
        return (self.scale * x_normalized) + self.shift

In [7]:
ln = LayerNormalize(embedding_dim=5)

out_normalized = ln(batch_example)

mean = out_normalized.mean(dim=-1, keepdim=True)   
var = out_normalized.var(dim=-1, keepdim=True, unbiased=False)   
print("Mean: \n",mean)
print("Variance: \n",var)

Mean: 
 tensor([[-1.4901e-08],
        [ 2.3842e-08]], grad_fn=<MeanBackward1>)
Variance: 
 tensor([[0.8000],
        [0.8000]], grad_fn=<VarBackward0>)
